# Tutorial: A Tiny Language Model (TLM)
> Created Mar 28 2026 for the FSU Course: *Machine Learning in Physics* <br>
> This is an update of a previous version of this notebook. This version describes decoder-only models, which are currently favored over encoder-decoder models for Large-Language Models (LLM).<br>
> Harrison B. Prosper<br>

## Introduction
The **transformer** neural network model coupled with a mechanism called **attention** [1], which we describe in detail in this notebook, revolutionized machine-based artificial intelligence (AI) and triggered the exponential rise of **large-language models** (LLMs). The seminal work on transformers [1] describes a model based on an encoder-decoder architecture. However, since 2019, 
LLM-based models such as GPT-3, GPT-4, Claude, Llama, Mistral, Gemma, Falcon, Command R, and Grok are based on **decoder-only** models. In this tutorial, we illustrate the construction of an LLM by building, from scratch, a **tiny language model** (TLM) based on a decoder-only architecture that can be trained in under two hours on a high-end GPU.
Many of the details used in this tutorial are taken from the excellent description of **transformers** in the Annotated Transformer[2]. (Note, however, the latter considers the more complex encoder-decoder model, which is no longer favored.) 

In a text-based LLM, which is what we consider here, the corpus of text that is used to train the model is broken up into **tokens**. From these tokens a set of tokens called a **vocabulary** is constructed, wherein each unique token is mapped to a unique integer.  Ideally, the vocabulary is broad enough to represent all text that the LLM is likely to encounter. The process of breaking text, or other media, into tokens is called **tokenization**. In this tutorial, we use a hand-coded tokenizer.

The model consists of three parts:

  1. The embedding layers embed the tokens and their relative positions within sequences into a vector space. A sequence of tokens is thus mapped to a point cloud in the vector space.
  1. The transformer layers[1] implement the syntactic and semantic analyses using a technique called **masked multi-head attention**.
  1. The output layer computes weights, called **logits**, one for every possible token in the vocabulary. 

The processing of a **prompt**, that is, the initial input source, proceeds **autoregressively**. 

  1. The prompt is sent to the model together with a separator token that indicates the end of the prompt.
  2. The model computes logits for every token in the vocabulary, which are converted to a probability distribution over the vocabulary, conditioned on the prompt and the separator token.
  3. The token with the highest probability is taken as the next token.
  4. That token is appended to the prompt and the separator tokens, which form the next input sequence into the model.
  5. The model continues until either the end-of-sequence token or the maximum number of output tokens is reached.

## Problem Statement
In this tutorial we solve the following problem: given a prompt, that is, a source sequence, $\boldsymbol{p} = f(x)$ constructed from pairs of trigonometric and hyperbolic functions, create a sequence to sequence (seq2seq) model that emits the Taylor series expansion of the function $f(x)$ to ${\cal O}(x^6)$.
In natural language translation, a word, for example $\texttt{the}$, or part of a word, for example $\texttt{ly}$, could be a token. In symbolic mathematics, a token might be a mathematical function, e.g., $\texttt{sin}$. In this tutorial the tokens are restricted to elements, such as, $($, $/$, $+$,  etc., which appear in simple mathematical expressions.

## Attention

When we translate from one sequence of symbols to another, for example from one natural language to another,  the meaning of the sequences is encoded in the symbols, their relative order, and the degree to which a given symbol is related to the other symbols. Consider the phrases "the white house" and "la maison blanche". In order to obtain a correct translation it is important for the model to encode the fact that "la" and "maison" are strongly related and that their order matters, and likewise for "the" and "house". It is also important for the model to encode the strong relationship between "the" and "la", between "house" and "maison", and between "white" and "blanche". Each token needs to *pay attention to* other tokens so that semantic and syntactic facts are correctly handled.

The need for the model to pay attention to relevant linguistic facts is the basis of the  [attention mechanism](https://nlp.seas.harvard.edu/annotated-transformer/). The model associates a vector to every token that captures the strength of a token's relationship to other tokens in the sequence. Since this association mechanism operates within the same sequence (that is, within the same point cloud in the vector space in which the sequence is embedded) it is referred to as **self attention**. The optimal way to implement this idea is not known, but the attention mechanism described in Ref. [1], described later in this notebook, and subsequent variants have proven to be highly effective.

## Prediction
For a vocabulary of size $m$ and a sequence of size $k$ every position in the sequence can be filled in $m$ ways. Therefore, there are $m^k$ possible sequences of which we want the most probable. Alas we have a bit of a computational problem. For example, given a sequence of size $k=85$ tokens and a vocabulary of size $m = 28$ tokens, we face the task of finding the most probable sequence from $\sim 10^{123}$ possible sequences. Even at a trillion probability calculations per second an exhaustive search would be an utterly futile undertaking because it would take far longer to complete than the current age of the universe ($\sim 4 \times 10^{17}$ s)! Obviously, we have no choice but to use a **heuristic strategy**.
The simplest such strategy, the one we shall use, is the **greedy search** in which we choose the most probable token as the next token. For deterministic applications, such as mathematics, this is the appropriate heuristic. 

**Tensor Convention**
The convention used in the Annotated Transformer[2] is followed in which the batch is the first dimension in all tensors. 

### References
  1.  [Attention is all you need](https://papers.nips.cc/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf)
  1. [Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/)

## Installation of `mlinphysics`

### Local installation
  ```bash
      git clone https://github.com/hbprosper/mlinphysics
      cd mlinphysics
      pip install -e .
  ```

## Running on Google Colab
If on Google Colab (https://colab.research.google.com), execute cell below.

In [1]:
try:
    import google.colab
    REPO = 'hbprosper/mlinphysics'
    
    url = f"https://raw.githubusercontent.com/{REPO}/refs/heads/main"
    !wget -q {url}/clone2colab.ipynb -O clone2colab.ipynb
    %run clone2colab.ipynb
    
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [2]:
import os, sys
import numpy as np
import importlib
import shutil

# PyTorch
import torch
import torch.nn as nn

# ML in physics module
import mlinphysics.nn as mlp
import mlinphysics.utils.data as dat
import mlinphysics.utils.monitor as mon
import mlinphysics.utils.tutorials as tut
import mlinphysics.utils.transformer as tnm

## Computational device

In [3]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nComputational device: {str(DEVICE):s}')


Computational device: cpu


In [4]:
# SEED = 42
# random.seed(SEED)
# np.random.seed(SEED)
# torch.manual_seed(SEED)
# torch.cuda.manual_seed(SEED)
# torch.backends.cudnn.deterministic = True

## Constants

In [5]:
DATAFILE = '../data/seq2seq_series_2terms.txt' 
MAX_SEQ_LEN = 128

# Model hyperparameters       
EMB_DIM = 72    # Dimension of embedding vector space
LAYERS  = 3     # Number of decoder layers
HEADS   = 8     # Number of decoder heads
FF_DIM  = 128
DROPOUT = 0.1

# Training hyperparameters
BATCH_SIZE    = 32
BASE_LR       = 16.0e-4           # Learning rate at the start
FACTOR_LR     = 16                # Final learning rate = BASE_LR / FACTOR_LR 
NSTEPS        = 16
NITERATIONS   = 800_000
STEP          = 100

## Read Sequence Data

The file **seq2seq_series_2terms.txt** contains (prompt, target) pairs where the targets are the Taylor series expansions of the corresponding prompts up to an error term of ${\cal O}(x^6)$ and the prompts, i.e., sources, are functions built from one or two terms randomly sampled from the set `{exp, sin, cos, tan, sinh, cosh, tanh}`. Since the source sequences are reasonably simple functions it is possible to train a transformer model to predict their Taylor series expansions in about hour on a GPU. The more complicated functions in the file **seq2seq_series.txt** require more time.

In [6]:
importlib.reload(tnm)
seqdata = tnm.SequenceData(DATAFILE, max_seq_len=MAX_SEQ_LEN)
seqdata.pprint(seqdata.sequences[0])

	reading prompt/target sequences

	sample size: 14367

0


cosh(a*x)**3 + tanh(b*x)

1 + b*x - b**3*x**3/3 + 2*b**5*x**5/15 + 3*a**2*x**2/2 + 7*a**4*x**4/8 + O(x**6)


4500


exp(a*x)*cosh(h*x)**2

...

-1 - m**2*x**2/2 - m**4*x**4/24 + c*x + c**3*x**3/6 + c**5*x**5/120 + O(x**6)


build vocabulary
{'<pad>': 0, '<sos>': 1, '<eos>': 2, '<sep>': 3, ' ': 4, '(': 5, ')': 6, '*': 7, '**': 8, '+': 9, '-': 10, '/': 11, '0': 12, '1': 13, '2': 14, '3': 15, '4': 16, '5': 17, '6': 18, '7': 19, '8': 20, '9': 21, 'O(x**6)': 22, 'a': 23, 'b': 24, 'c': 25, 'cos': 26, 'cosh': 27, 'd': 28, 'exp': 29, 'f': 30, 'g': 31, 'h': 32, 'm': 33, 'n': 34, 'sin': 35, 'sinh': 36, 'tan': 37, 'tanh': 38, 'x': 39}

tokenize prompts and targets
 14000
 14000
concatenate prompts and targets
pad sequences and bracket with <sos> and <eos>

Summary
 sample size:                   12956
  avg(sequence length):          68.2
  stdv(sequence length):         20.6
 vocabulary size:                  40

Sequence
[ 1 27  5 23  7 39  6  8 15  9 38  5 24  7 39  6  3 13  9 24  7 39  9 15
  7 23  8 14  7 39  8 14 11 14 10 24  8 15  7 39  8 15 11 15  9 19  7 23
  8 16  7 39  8 16 11 20  9 14  7 24  8 17  7 39  8 17 11 13 17  9 22  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  

cosh(a*x)**3 + tanh(b*x)

Target


1 + b*x - b**3*x**3/3 + 2*b**5*x**5/15 + 3*a**2*x**2/2 + 7*a**4*x**4/8 + O(x**6)

## Configuration

In [7]:
sequences = seqdata.sequences
ndata     = len(sequences)
train_size= 12_000
test_size =    800
val_size  = ndata - train_size - test_size

In [8]:
importlib.reload(mlp)
# ----------------------------------------
# Name of model
# -----------------------------------------
name = 'tinyLM'

# Create new configuration object
config = mlp.Config(name, dirname=name)
# ----------------------------------------
# Training configuration
# ----------------------------------------
config('train_size',  train_size) # Training dataset size
config('val_size',    val_size)
config('test_size',   test_size)
config('batch_size',  BATCH_SIZE) # Number of graphs / batch
config('monitor_step',STEP)       # Monitor training every n (=10) iterations
config('frac', 0.01)              # Save model if average loss decreases by
                                  # More than a fraction "frac"
# ----------------------------------------
# Optimizer / scheduler configuration
# ----------------------------------------
# A step comprises a given number of iterations
config('n_steps', NSTEPS)         # Number of training steps
GAMMA = (1/FACTOR_LR)**(1/(NSTEPS-1)) if NSTEPS > 1 else 1.0
config('gamma', GAMMA)            # Learning rate scale factor
config('n_iterations', NITERATIONS)
config('n_iters_per_step', int(config('n_iterations') / config('n_steps')))
config('base_lr', BASE_LR)        # Initial learning rate
config('factor_lr', FACTOR_LR)    # Final lr = base_lr / factor_lr
# ----------------------------------------
# Data
# ----------------------------------------
config('DATAFILE',    DATAFILE)
config('MAX_SEQ_LEN', MAX_SEQ_LEN)
# ----------------------------------------
# Model specification
# ----------------------------------------
config('EMB_DIM', EMB_DIM)        # Dimension of embedding vector space
config('LAYERS',  LAYERS)         # Number of encoder layers
config('HEADS',   HEADS)          # Number of attention heads
config('FF_DIM',  FF_DIM)         # "hidden" dimension of ff-network
config('DROPOUT', DROPOUT)

config('VOCAB_SIZE', seqdata.VOCAB_SIZE)

config('PAD', seqdata.PAD)  
config('SOS', seqdata.SOS)
config('EOS', seqdata.EOS)
config('SEP', seqdata.SEP)
config('PROMPT_MASK', False)

print('\n\tConfiguration\n')
print(config)
print(f'\nSave configuration to file {config("file/config")}\n')

config.save()


	Configuration

name: tinyLM
file:
  config: runs/tinyLM/tinyLM_config.yaml
  losses: runs/tinyLM/tinyLM_losses.csv
  script: runs/tinyLM/tinyLM_script.pth
  params: runs/tinyLM/tinyLM_params.pth
  init_params: runs/tinyLM/tinyLM_init_params.pth
  plots: runs/tinyLM/tinyLM_plots.png
train_size: 12000
val_size: 156
test_size: 800
batch_size: 32
monitor_step: 100
frac: 0.01
n_steps: 16
gamma: 0.8312378961427878
n_iterations: 800000
n_iters_per_step: 50000
base_lr: 0.0016
factor_lr: 16
DATAFILE: ../data/seq2seq_series_2terms.txt
MAX_SEQ_LEN: 128
EMB_DIM: 72
LAYERS: 3
HEADS: 8
FF_DIM: 128
DROPOUT: 0.1
VOCAB_SIZE: 40
PAD: 0
SOS: 1
EOS: 2
SEP: 3
PROMPT_MASK: false


Save configuration to file runs/tinyLM/tinyLM_config.yaml



## Datasets

In [9]:
importlib.reload(dat)

train_size = config('train_size')
val_size   = config('val_size')
test_size  = config('test_size')

# training dataset (this defines the empirical risk to be minimized)
print('training data')
train_data = dat.Dataset(
    sequences, start=0, end=train_size)

# a random subset of the training data to check for overtraining
# by comparing with the empirical risk from the validation set
print('training data for validation')
train_data_val = dat.Dataset(
    sequences, start=0, end=train_size, random_sample_size=val_size)

# validation dataset (for monitoring training)
print('validation data')
val_data = dat.Dataset(
    sequences, start=train_size, end=train_size + val_size)

# test dataset
print('test data')
test_data= dat.Dataset(sequences,
                       start=train_size + val_size,
                       end=train_size + val_size + test_size)

training data
Dataset
  shape of x: torch.Size([12000, 128])

training data for validation
Dataset
  shape of x: torch.Size([156, 128])

validation data
Dataset
  shape of x: torch.Size([156, 128])

test data
Dataset
  shape of x: torch.Size([800, 128])



## DataLoaders

In [10]:
importlib.reload(dat)

print('train data loader')
train_loader = dat.DataLoader(train_data, 
                              batch_size=config('batch_size'),
                              num_iterations=config('n_iterations'))

print('train data loader for validation')
train_loader_val = dat.DataLoader(train_data_val, 
                                  batch_size=len(train_data_val))

print('validation data loader')
val_loader = dat.DataLoader(val_data, 
                            batch_size=len(val_data))

print('test data loader')
test_loader = dat.DataLoader(test_data, 
                             batch_size=1)

train data loader
DataLoader
  Number of iterations has been specified
  maxiter:          800000
  batch_size:           32
  shuffle_step:        375

train data loader for validation
DataLoader
  maxiter:               1
  batch_size:          156
  shuffle_step:          1

validation data loader
DataLoader
  maxiter:               1
  batch_size:          156
  shuffle_step:          1

test data loader
DataLoader
  maxiter:             800
  batch_size:            1
  shuffle_step:        800



## The Decoder

The **decoder** does the following:
 1. Each token in a sequence, $\boldsymbol{X}$, is encoded as a vector $\boldsymbol{t}$ in a space of $d =$ **emb_dim** dimensions. A sequence is therefore represented as a point cloud in the vector space.
 1. The position of each token is also encoded as a vector $\boldsymbol{z}$ in a vector space of the same dimension as $\boldsymbol{t}$. We can think of $\boldsymbol{z}$ as residing in the same vector space as the vector $\boldsymbol{t}$.  Both the token and position embeddings are trainable.
 1. Each token is associated with a third vector: $\boldsymbol{v} = \nu \, \boldsymbol{t} + \boldsymbol{z}$, where the scale factor $\nu$  in this tutorial is a trainable parameter. A token and its position are encoded as a point on the line defined by the vectors $\boldsymbol{t}$ and $\boldsymbol{z}$.
 2. The vectors $\boldsymbol{v}$ are processed through $N$ *decoder layers*.

Since the sequences are **padded** so that they are all of equal length, a method is needed to ensure that the pad tokens are ignored in all calculations. This is done using **masks**. Any token that needs to be ignored
is identified with a zero in the `mask`. In the masked multi-head attention calculation (see below) all such tokens are ignored by replacing values in certain calculations with a large negative number $(-10^{10})$ so that when a softmax operation is performed on the results of these calculations the contribution from the associated tokens is zero. (This is explained in detail later.)

A **decoder layer** does the following:

 1. The embedded sequence and its mask are passed to a masked multi-head attention layer.
 1. A residual connection and [Layer Normalization](https://arxiv.org/abs/1607.06450) is applied.
 1. A linear layer is applied.
 1. And finally a residual connection and layer normalization is applied.


**Note**: The TLM codes below are fully type annotated to help the PyTorch scripting tool avoid mistakes. We use the scripting tool (`torch.jit.script`) to  create a representation of a (trained) model that can be saved to a file. 

In [11]:
class Decoder(nn.Module):
    
    def __init__(self, 
                 vocab_size : int,   # size of vocabulary
                 max_len :    int,   # maximum sequence length
                 emb_dim :    int,   # dimension of embedding vector space
                 n_layers :   int,   # number of decoder layers
                 n_heads :    int,   # number of masked attention heads
                 ff_dim :     int,   # hidden dimension of feed-forward network
                 dropout :    float, # dropout probability
                 device :     torch.device) -> None: # computational device
        
        super().__init__()

        self.device = device

        # Represent each of the 'vocab_size' tokens by a vector 
        # of size d = emb_dim. nn.Embedding "learns" a simple
        # lookup table that maps the code for each token in the
        # vocabulary to a vector.
        self.tok_embedding = nn.Embedding(vocab_size, emb_dim)

        # Represent the position of each token by a vector of 
        # size d = emb_dim.
        # 'max_len' is the maximum length of a sequence.
        self.pos_embedding = nn.Embedding(max_len, emb_dim)

        # Create decoding layers
        self.layers  = nn.ModuleList([DecoderLayer(emb_dim, 
                                                   n_heads, 
                                                   ff_dim, 
                                                   dropout, 
                                                   device)
                                     for _ in range(n_layers)])

        # Layer to map processed token vectors to logits over the vocabulary
        self.linear  = nn.Linear(emb_dim, vocab_size)

        # Randomly set to zero weights during training.
        # Dropout is thought to mitigate over-training.
        self.dropout = nn.Dropout(dropout)

        # Factor by which to scale token embedding vectors.
        # use nn.Parameter to tell PyTorch that this is a 
        # trainable parameter.
        self.nu = nn.Parameter(torch.sqrt(torch.FloatTensor([emb_dim])))
 
    def forward(self, 
                sequence : torch.Tensor, 
                mask : torch.Tensor) -> torch.Tensor:
        # sequence : [batch_size, seq_len]
        # mask     : [batch_size, 1, seq_len, seq_len]
        #                         ^---- axis for attention heads        
        batch_size, seq_len = sequence.shape

        # ---------------------------------------
        # Token embedding 
        # ---------------------------------------
        token = self.tok_embedding(sequence)
        # token: [batch_size, seq_len, emb_dim]
        
        # ---------------------------------------
        # Token position embedding
        # ---------------------------------------
        # Create a row tensor, position (=z), with entries [0, 1,..., seq_len-1]
        position = torch.arange(0, seq_len)
        # position: [seq_len]
        
        # 1. Use unsqueeze(0) to add a dimension (for the batch)
        #    so that the position tensor will have shape [1, seq_len].
        position = position.unsqueeze(0)
        
        # 2. Repeat one instance of the position tensor per row 
        #    'batch_size' times so that we get:
        # position = |position|
        #            |position|
        #                :
        #            |position|
        once_per_row = 1
        position = position.repeat(batch_size, once_per_row)
        # position: [batch_size, seq_len]
        
        # 3. Send to computational device
        position = position.to(self.device)
        # position: [batch_size, seq_len]
        
        # 4. Embed position (ordinal value of token) in a vector space.
        position = self.pos_embedding(position)
        # position: [batch_size, seq_len, emb_dim]

        # 5. Represent a token and its position as a point on a
        #    line within the vector space.
        #    (Perhaps, this could be generalized using an MLP?)
        seq = self.nu * token + position
        # seq: [batch_size, trg_len, emb_dim]

        # This seems to be helpful!
        seq = self.dropout(seq)
        
        # Process the embedded sequences through a series of layers. 
        for layer in self.layers:
            seq = layer(seq, mask)
            # seq: [batch_size, seq_len, emb_dim]

        # For each token, output 'vocab_size' logits, one for each
        # token in the vocabulary. The logits will be later converted 
        # to probabilities. This is done, for example, in the CrossEntropyLoss
        logits = self.linear(seq)
        # logits: [batch_size, seq_len, vocab_size]
            
        return logits

### Multi-Head Attention Layer, the $(A,B,C)$ parametrization

We start from the same definition of attention as used in the notebook `tutorial_tiny_langugae_model`,
\begin{align}
    \texttt{Attention}(Q, K, V) & = \texttt{softmax}\left(\mu Q K^T \right) V,
\end{align}
with $Q = xA_Q^T + b_Q$, $K = xA_K^T + b_K$, $V = xA_V^T + b_V$. Expanding $QK^T$ in terms of $x$ gives four terms: one quadratic in $x$, two linear in $x$, and one constant,
\begin{align}
    Q_iK_j^T & = x_iA_Q^TA_Kx_j^T + x_iA_Q^Tb_K^T + b_QA_Kx_j^T + b_Qb_K^T .
\end{align}
Because $\texttt{softmax}$ is applied along the key axis $j$ (row-wise) and is invariant to adding a constant to an entire row, every term above that is independent of $j$ does not contribute: that includes $x_iA_Q^Tb_K^T$ (it depends on $i$, not $j$) and $b_Qb_K^T$ (it depends on neither). Both of these terms are proportional to the key bias, $b_K$, which is independent of $j$, so $b_K$ can be dropped from the model without changing a single attention weight it computes. 

The one bias-like term that survives, $b_QA_Kx_j^T$, is the only place the query bias $b_Q$ matters, and note that it is really just a *single fixed vector*, $c^T \equiv b_QA_K \in \mathbb{R}^{1\times h}$, dotted against every token $x_j$ — there is nothing further to be gained by forcing it through the product of a bias and a weight matrix that is *also* used for the quadratic term. We give it its own name and parametrize it directly.

Writing $A \equiv A_Q$, $B \equiv A_K$ (both bias-free — recall neither $A$ nor $B$ is individually identifiable, only the product $W = A^TB$ is, so nothing is lost by dropping their biases) and $C \in \mathbb{R}^{1\times h}$ for the surviving bias direction, the pre-softmax score is
\begin{align}
    S & = \mu\left[x A^T B x^T + C x^T\right],
\end{align}
and, since $\texttt{softmax}$ removes nothing else,
\begin{align}
    \texttt{Attention}(x) & = \texttt{softmax}\!\left(\mu\left[x A^T B x^T + C x^T\right]\right) V(x), \qquad V(x) = xA_V^T + b_V .
\end{align}
This is exactly the same family of functions computed by the original $Q/K/V$ formula — nothing has been lost — but it is now written using only the parameters that actually affect the output: a rank-$\le h_{\text{head}}$ bilinear form $W = A^TB$ on token pairs (playing the geometric role described above: $\texttt{softmax}$ maps each row onto the simplex spanned by the tokens, as in the figure), a single linear functional $C$ of the key tokens, and the value map $V$. There is no real sense left in which $A$ and $B$ are separately a "query" and a "key" — they are the two low-rank factors of one bilinear form — so from here on we just call them $A$ and $B$.

#### Building each head as a separate object

For pedagogical clarity, each head $\ell = 1,\dots,\text{n\_heads}$ of the multi-head attention is built as its *own* small module, owning its own $A_\ell, B_\ell, C_\ell, V_\ell, \mu_\ell$:
\begin{align}
    \texttt{head}_\ell(x) & = \texttt{softmax}\!\left(\mu_\ell\left[xA_\ell^TB_\ell x^T + C_\ell x^T\right]\right)V_\ell(x), \qquad
    A_\ell, B_\ell, V_\ell : \mathbb{R}^h \to \mathbb{R}^{h_{\text{head}}},\;\; C_\ell : \mathbb{R}^h \to \mathbb{R} ,
\end{align}
and the `n_heads` outputs, each of shape `head_dim`, are concatenated into a single vector of dimension $h$ = `emb_dim`, exactly as in Steps 6–7 of the original algorithm, before being passed through the same output layer $O$ (Step 8). The two constructions — fused-then-reshaped, and separate-then-concatenated — compute *exactly* the same family of functions; building the heads separately is purely a matter of making the object that each head owns, and its parameter count ($h_{\text{head}}\times h$ for $A_\ell$ and $B_\ell$, $1\times h$ for $C_\ell$, $h_{\text{head}}\times h$ plus a bias of size $h_{\text{head}}$ for $V_\ell$), visible directly in the code rather than implicit in a reshape.

In [12]:
from typing import List

class AttentionHead(nn.Module):
    '''
    A single attention head, computing

        softmax( mu * [ x A^T B x^T + C x^T ] ) V(x)

    with V(x) = x A_V^T + b_V.

    emb_dim  : int    Embedding dimension (=h)
    head_dim : int    Dimension of this head's sub-vectors
    dropout  : float  Dropout probability

    A, B : nn.Linear(emb_dim, head_dim, bias=False)
        The two low-rank factors of the pairwise bilinear form
        W = A^T B (rank <= head_dim). Neither is individually
        identifiable -- only their product is -- and, as shown
        above, softmax kills every term built from a key-side
        bias, so there is no bias to give either of them.

    C : nn.Linear(emb_dim, 1, bias=False)
        Produces the single surviving bias-like term, the scalar
        c_j = C x_j for each token j, broadcast identically across
        every query row i. This directly replaces the product
        b_Q A_K from the original Q/K formulation with a single,
        directly-trained (1, emb_dim) vector.

    V : nn.Linear(emb_dim, head_dim, bias=True)
        The value projection. Its bias is *not* touched by softmax
        (softmax's rows sum to one, so a constant shift in V is
        just carried through as a constant shift in the output) so,
        unlike a key bias, it is a genuine, non-redundant parameter
        and is kept.

    mu : trainable scale factor, one per head, initialized to
        1 / sqrt(head_dim) (the actual contraction dimension for
        this head, rather than emb_dim).
    '''

    def __init__(self,
                 emb_dim  : int,
                 head_dim : int,
                 dropout  : float) -> None:

        super().__init__()

        self.A = nn.Linear(emb_dim, head_dim, bias=False)
        self.B = nn.Linear(emb_dim, head_dim, bias=False)
        self.C = nn.Linear(emb_dim, 1,        bias=False)
        self.V = nn.Linear(emb_dim, head_dim, bias=True)

        self.dropout = nn.Dropout(dropout)

        # Trainable scale factor for this head.
        self.mu = nn.Parameter(1 / torch.sqrt(torch.FloatTensor([head_dim])))

    def forward(self,
                sequence : torch.Tensor,
                mask     : torch.Tensor) -> torch.Tensor:
        # sequence : [batch_size, seq_len, emb_dim]
        # mask     : [batch_size, seq_len, seq_len]   (heads dim already squeezed out)

        Ax = self.A(sequence)   # [batch_size, seq_len, head_dim]
        Bx = self.B(sequence)   # [batch_size, seq_len, head_dim]
        Vx = self.V(sequence)   # [batch_size, seq_len, head_dim]

        # Quadratic term: x A^T B x^T, computed as Ax @ Bx^T
        S = torch.matmul(Ax, Bx.transpose(-2, -1))
        # S: [batch_size, seq_len, seq_len]

        # Linear term: c_j = C x_j, broadcast across every query row i.
        c = self.C(sequence)          # [batch_size, seq_len, 1]
        c = c.transpose(-2, -1)       # [batch_size, 1, seq_len]
        S = S + c
        # S: [batch_size, seq_len, seq_len]

        S = self.mu * S

        # Same masking convention as before: replace disallowed
        # entries with a large negative number before softmax.
        S = S.masked_fill(mask == 0, -1e10)

        # W lies, row-wise, in the simplex (see figure above).
        W = torch.softmax(S, dim=-1)
        W = self.dropout(W)

        attention = torch.matmul(W, Vx)
        # attention: [batch_size, seq_len, head_dim]

        return attention


class MultiHeadAttentionPerHead(nn.Module):

    def __init__(self,
                 emb_dim : int,
                 n_heads : int,
                 dropout : float,
                 device  : torch.device) -> None:
        '''
    Pedagogical version: every head is its own object (see AttentionHead
    above). Mathematically identical to, but much slower than,
    MultiHeadAttention below, which batches all heads into a handful of
    large matrix multiplications instead of looping over small ones.

    emb_dim : int    Embedding dimension (=h)
    n_heads : int    Number of attention heads
    dropout : float  Dropout probability
    device  :        Computational device (kept only for interface
                      consistency with the rest of the code -- it
                      is not needed explicitly here)

    Unlike the fused-then-reshaped implementation, each head is
    built as its own AttentionHead submodule, with its own A, B,
    C, V, and mu. The two constructions compute exactly the same
    family of functions -- concatenating n_heads independent
    rank-head_dim bilinear heads is precisely what the fused,
    split-by-reshaping version computes -- but here every head's
    parameters are separate, literal objects.
    '''

        super().__init__()

        # emb_dim must be a multiple of n_heads
        assert emb_dim % n_heads == 0

        self.emb_dim  = emb_dim
        self.n_heads  = n_heads

        # Compute dimension of sub-vectors
        self.head_dim = emb_dim // n_heads

        # One separate, self-contained attention head per element.
        self.heads = nn.ModuleList(
            [AttentionHead(emb_dim, self.head_dim, dropout)
             for _ in range(n_heads)])

        self.O = nn.Linear(emb_dim, emb_dim) # Output

    def forward(self,
                sequence : torch.Tensor,
                mask : torch.Tensor) -> torch.Tensor:
        # sequence : [batch_size, seq_len, emb_dim]
        # mask     : [batch_size, 1, seq_len, seq_len]
        #                         ^---- heads dimension

        batch_size, seq_len, emb_dim = sequence.shape
        assert emb_dim == self.emb_dim

        # All heads share the same mask; drop the singleton
        # heads dimension once, up front.
        head_mask = mask.squeeze(1)
        # head_mask: [batch_size, seq_len, seq_len]

        outputs : List[torch.Tensor] = []
        for head in self.heads:
            outputs.append(head(sequence, head_mask))
        # each entry of outputs: [batch_size, seq_len, head_dim]

        # Concatenate the n_heads outputs into a single tensor.
        # This plays the same role as Steps 6-7 (permute + view)
        # in the original fused-then-reshaped algorithm.
        attention = torch.cat(outputs, dim=-1)
        # attention: [batch_size, seq_len, emb_dim]

        logits = self.O(attention)
        # logits: [batch_size, seq_len, emb_dim]

        return logits


### The same $(A,B,C)$ attention, computed the efficient way

`MultiHeadAttentionPerHead` above is deliberately literal: each head is a separate Python object with its own small `nn.Linear` layers, and the `n_heads` heads are run one at a time in a Python `for` loop. That is exactly why it is slow — on a GPU, a handful of large matrix multiplications is far faster than the same total amount of arithmetic split across many small ones launched sequentially from Python, since each small matmul pays a fixed kernel-launch overhead and can't use the hardware's parallelism nearly as well as one big batched multiply.

The fix is the same trick used in the original fused $Q/K/V$ algorithm (Steps 1–8 above): stack every head's $A_\ell, B_\ell, C_\ell, V_\ell$ into one large linear layer per parameter, apply it *once* to the whole sequence, and then split the result into `n_heads` blocks by reshaping. Concretely,

* $A$ and $B$ each become a single `nn.Linear(emb_dim, emb_dim, bias=False)` — the same shape as the original $f_Q$, $f_K$ — whose output, once reshaped into `n_heads` blocks of size `head_dim`, gives $A_\ell(x)$ and $B_\ell(x)$ for every head $\ell$ simultaneously.
* $V$ becomes a single `nn.Linear(emb_dim, emb_dim, bias=True)`, exactly as in the original algorithm.
* $C$ becomes a single `nn.Linear(emb_dim, \text{n\_heads}, bias=False)`: one scalar output per head, per token, rather than one output per head from `n_heads` separate size-1 linear layers.
* $\mu$ becomes a length-`n_heads` trainable vector (one scale factor per head) instead of `n_heads` separate scalars.

The computation performed is *exactly* the one derived above — for matching weights, this version and `MultiHeadAttentionPerHead` produce bit-for-bit identical output (only the bookkeeping of how the parameters are stored and multiplied differs) — but it now uses the same eight-step batched algorithm as the original $Q/K/V$ implementation:

1. Compute $A(x)$, $B(x)$, $V(x)$ once, each of shape **[batch_size, seq_len, emb_dim]**.
2. Split the last dimension into `n_heads` blocks of size `head_dim` and permute to **[batch_size, n_heads, seq_len, head_dim]**.
3. Transpose $B$'s last two dimensions to model $B^T$.
4. Compute the quadratic term $A(x)B(x)^T$ for every head at once via a single batched `torch.matmul`, giving a **[batch_size, n_heads, seq_len, seq_len]** tensor.
5. Compute $C(x)$, of shape **[batch_size, seq_len, n_heads]**, and reshape it to broadcast one scalar per head across every query row, then add it to the quadratic term and scale by $\mu$ (one value per head).
6. Mask and apply softmax along the key axis, exactly as before.
7. Multiply by $V(x)$ and permute/reshape the `n_heads` blocks of size `head_dim` back into a single **[batch_size, seq_len, emb_dim]** tensor.
8. Push the result through the output layer $O$.



In [13]:
class MultiHeadAttention(nn.Module):

    def __init__(self,
                 emb_dim : int,
                 n_heads : int,
                 dropout : float,
                 device  : torch.device) -> None:
        '''
    Efficient version of the (A,B,C) attention: mathematically identical
    to MultiHeadAttentionPerHead, but all n_heads heads are computed at
    once via a handful of large, batched matrix multiplications instead
    of a Python loop over n_heads small ones.

    emb_dim : int    Embedding dimension (=h)
    n_heads : int    Number of attention heads
    dropout : float  Dropout probability
    device  :        Computational device (kept for interface
                      consistency; not needed explicitly here)
    '''

        super().__init__()

        # emb_dim must be a multiple of n_heads
        assert emb_dim % n_heads == 0

        self.emb_dim  = emb_dim
        self.n_heads  = n_heads

        # Compute dimension of sub-vectors
        self.head_dim = emb_dim // n_heads

        # A and B are the (bias-free) low-rank factors of the n_heads
        # pairwise bilinear forms, stacked head-by-head along the
        # output dimension -- exactly like fQ, fK in the original code.
        self.A = nn.Linear(emb_dim, emb_dim, bias=False)
        self.B = nn.Linear(emb_dim, emb_dim, bias=False)

        # One scalar output per head, per token: C plays the role of
        # the n_heads separate (1, emb_dim) vectors, stacked into a
        # single (n_heads, emb_dim) weight matrix.
        self.C = nn.Linear(emb_dim, n_heads, bias=False)

        # Value projection, stacked head-by-head, same as the
        # original fV. Its bias survives softmax (see above) and
        # is kept.
        self.V = nn.Linear(emb_dim, emb_dim, bias=True)

        self.O = nn.Linear(emb_dim, emb_dim) # Output

        self.dropout = nn.Dropout(dropout)

        # One trainable scale factor per head, initialized using
        # each head's own contraction dimension, head_dim.
        self.mu = nn.Parameter(
            torch.ones(n_heads, 1, 1) / (self.head_dim ** 0.5))

    def split(self, t : torch.Tensor) -> torch.Tensor:
        # t: [batch_size, seq_len, emb_dim]
        batch_size, seq_len, _ = t.shape
        t = t.view(batch_size, seq_len, self.n_heads, self.head_dim)
        return t.permute(0, 2, 1, 3)
        # : [batch_size, n_heads, seq_len, head_dim]

    def forward(self,
                sequence : torch.Tensor,
                mask : torch.Tensor) -> torch.Tensor:
        # sequence : [batch_size, seq_len, emb_dim]
        # mask     : [batch_size, 1, seq_len, seq_len]
        #                         ^---- heads dimension (broadcasts below)

        batch_size, seq_len, emb_dim = sequence.shape
        assert emb_dim == self.emb_dim

        # Step 1: compute A(x), B(x), V(x) once for all heads.
        Ax = self.A(sequence)   # [batch_size, seq_len, emb_dim]
        Bx = self.B(sequence)   # [batch_size, seq_len, emb_dim]
        Vx = self.V(sequence)   # [batch_size, seq_len, emb_dim]

        # Steps 2-3: split into n_heads blocks of size head_dim and
        # move the heads dimension next to the batch dimension.
        Ax = self.split(Ax)     # [batch_size, n_heads, seq_len, head_dim]
        Bx = self.split(Bx)     # [batch_size, n_heads, seq_len, head_dim]
        Vx = self.split(Vx)     # [batch_size, n_heads, seq_len, head_dim]

        # Step 4: quadratic term A(x) B(x)^T, for every head at once.
        S = torch.matmul(Ax, Bx.transpose(-2, -1))
        # S: [batch_size, n_heads, seq_len, seq_len]

        # Step 5: linear term, one scalar per head per key token j,
        # broadcast across every query row i.
        c = self.C(sequence)                # [batch_size, seq_len, n_heads]
        c = c.permute(0, 2, 1).unsqueeze(2) # [batch_size, n_heads, 1, seq_len]
        S = S + c

        # mu: [n_heads, 1, 1] broadcasts against
        # S: [batch_size, n_heads, seq_len, seq_len]
        S = self.mu * S

        # Step 6: mask (broadcasts over the heads dimension) and softmax.
        S = S.masked_fill(mask == 0, -1e10)
        W = torch.softmax(S, dim=-1)
        W = self.dropout(W)

        # Step 7: multiply by V(x) and merge the heads back together.
        attention = torch.matmul(W, Vx)
        # attention: [batch_size, n_heads, seq_len, head_dim]

        attention = attention.permute(0, 2, 1, 3).contiguous()
        attention = attention.view(batch_size, seq_len, self.emb_dim)
        # attention: [batch_size, seq_len, emb_dim]

        # Step 8: output layer.
        logits = self.O(attention)
        # logits: [batch_size, seq_len, emb_dim]

        return logits


### Feedforward Layer

In [14]:
class Feedforward(nn.Module):
    
    def __init__(self, 
                 emb_dim : int, 
                 ff_dim  : int, 
                 dropout : float) -> None:
        '''
    emd_dim: int    Embeding dimension (=d)
    ff_dim: int     Number of nodes in hidden layer
    dropout: float  Dropout probability
        '''
       
        super().__init__()
        
        self.linear_1 = nn.Linear(emb_dim, ff_dim)
        
        self.linear_2 = nn.Linear(ff_dim, emb_dim)
        
        self.dropout  = nn.Dropout(dropout)

        self.silu     = nn.SiLU()
        
    def forward(self, x : torch.Tensor) -> torch.Tensor:
        # x: [batch_size, seq_len, emb_dim]
        
        x = self.linear_1(x)
        # x: [batch_size, seq_len, ff_dim]
        
        x = self.silu(x)
        
        x = self.dropout(x)
        
        x = self.linear_2(x)
        # x: [batch_size, seq_len, emb_dim]
        
        return x

### Decoder Layer

Each decoder layer has a multi-head self attention layer.

In [15]:
class DecoderLayer(nn.Module):
    
    def __init__(self, 
                 emb_dim : int, 
                 n_heads : int, 
                 ff_dim  : int, 
                 dropout : float, 
                 device  : torch.device) -> None:
        '''
    emd_dim: int    Embeding dimension (=d)
    n_heads: int    Number of attention heads
    ff_dim:  int    Number of nodes in hidden layer
    dropout: float  Dropout probability
    device:         Computational device
        '''

        super().__init__()

        # Attention mechanism
        self.self_attention      = MultiHeadAttention(
            emb_dim, n_heads, dropout, device)
        
        self.self_attention_norm = nn.LayerNorm(emb_dim)
        
        self.feedforward         = Feedforward(emb_dim, ff_dim, dropout)
        
        self.feedforward_norm    = nn.LayerNorm(emb_dim)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, 
                sequence : torch.Tensor, 
                mask : torch.Tensor) -> torch.Tensor:
        # sequence : [batch_size, seq_len, emb_dim]
        # mask     : [batch_size, 1, seq_len, seq_len]
        #                         ^---- heads dimension
        
        # Compute attention over sequences.
        # Distinguish between sequence and seq_, since the former 
        # is needed later for residual connections.            
        seq_ = self.self_attention(sequence, mask)
        # seq_: [batch_size, seq_len, emb_dim]

        # This could be helpful
        seq_ = self.dropout(seq_)
        
        # Add residual connection and layer norm 
        seq  = self.self_attention_norm(sequence + seq_)
        # seq: [batch_size, seq_len, emb_dim]

        # Again distinguish between seq and seq_
        seq_ = self.feedforward(seq)
        # seq_: [batch_size, seq_len, emb_dim]

        # Randomly drop some elements in seq_
        seq_ = self.dropout(seq_)
        
        # Add another residual connection and layer norm
        seq  = self.feedforward_norm(seq + seq_)
        # seq: [batch_size, seq_len, emb_dim]
        
        return seq

## The `transformer` Model

The `transformer` decoder-only model, which is typical of LLMs circa 2026, encapsulates the decoder and the creation of 
the pad and subsequent (aka causal) masks. Both masks are described below.

In the pad mask a `<pad>` token is identified with a 0 and other tokens with a 1. The mask is reshaped so that it can be broadcast to tensors of shape **[batch_size, n_heads, seq_len, seq_len]** which appear in the masked multi-head attention calculation. This is necessary so that the same mask can be applied to every attention head. The mask is used to identify which tokens need to be ignored, namely, those for which the mask element is 0.

Consider a sequence $\boldsymbol{X} = \text{<sos>}, \boldsymbol{p}, \text{<sep>}, t_1,\cdots, t_{k}, \text{<eos>}$, where $\boldsymbol{p}$ denotes the sequence of prompt tokens and $t_i$ are the tokens to be predicted. The tokens $\text{<sos>}$, $\text{<sep>}$,  and $\text{<eos>}$, are the start-of-sequence, separator,  and end-of-sequence tokens, respectively. During training, ideally, for every sub-sequence, $\boldsymbol{X}_i$, we would like to predict the next token and test the quality of the prediction for all sub-sequences *simultaneously*. For example, given sub-sequences $\text{<sos>}, \boldsymbol{p}, \text{<sep>}$ and
$\text{<sos>}, \boldsymbol{p}, \text{<sep>}, t_1$, 
we would like to check simultaneously the predictions $\text{<sos>}, \boldsymbol{p}, \text{<sep>} \rightarrow y_1$ and $\text{<sos>}, \boldsymbol{p}, \text{<sep>}, t_1 \rightarrow y_2$, where $y_1$ and $y_2$ are the model predictions for the next token associated with each sub-sequence and $t_1$ and $t_2$ are the corresponding correct target tokens. 

In practice, the decoder emits vectors of weights, $\ell_i$, called **logits**, of dimension equal to the size $|\mathbb{V}|$ of the vocabulary, $\mathbb{V}$, which are converted to a discrete conditional probability distribution 
\begin{align}
p(t_{i+1} \in \mathbb{V}| \boldsymbol{X}_i).
\end{align}
The most probable token is taken to be the next output token, $y_i$.

A crucial advantage of a transformer compared with earlier sequence models is that during training, the model computes logits for all the sub-sequences of a sequence, $\boldsymbol{X}$, *in parallel*. This is achieved with a simple, but clever, trick: the **subsequent** or **causal mask**, `sub_mask`. This mask, created using  the function $\texttt{torch.tril}$, is a lower diagonal square matrix where the elements above the diagonal are zero and all other elements are unity. Just before the application of the softmax in the attention code,  the elements identified by the zeros in `sub_mask` of the tensor that enters the softmax are replaced with a large negative number. This causes the masked tokens to contribute zero when the softmax is applied, thereby ensuring that the calculations on a sub-sequence depend only on the tokens of the sub-sequence. For a given sub-sequence, the model cannot "cheat" by looking ahead at the correct next token, that is, cannot "attend to" subsequent tokens. Since the pad tokens are also masked, they too cannot contribute to the attention calculation. 

Consider, for example, the sequence $\boldsymbol{X} = \text{<sos>}, p, \text{<sep>}, t_1, \text{<eos>}$ comprising 5 tokens.  The semi-causal mask, called `sub_mask` in the code, looks like this:

$$\begin{pmatrix}
1 & 0 & 0 & 0 & 0\\
1 & 1 & 0 & 0 & 0\\
1 & 1 & 1 & 0 & 0\\
1 & 1 & 1 & 1 & 0\\
1 & 1 & 1 & 1 & 1\\
\end{pmatrix}.$$

When applied to the sequence of tokens, $\boldsymbol{X}$, the causal (or subsequent) mask, `sub_mask`, ensures that for every token the model has access only to the token and its predecessors. In other words, for a given sub-sequence, the tokens can attend to only tokens within the sub-sequence. For example, the first row of the causal mask is **[1, 0, 0, 0, 0]**. The sub-sequence $\text{<sos>}$ can attend to itself only, so the model is forced to predict the next token given the sequence $\text{<sos>}$. (At this stage, the model would be hard-pressed to do so!) The second row of the causal mask is **[1, 1, 0, 0, 0]**. In this case, the tokens  $\text{<sos>}$ and $p$ can attend to each other but not to the subsequent tokens. Consequently, the model is forced to predict the next token. The same holds true for the remaining sub-sequences. Again it should be stressed that all of these calculations are done in parallel. 

The overall mask is the logical AND of the pad and causal masks.
Moreover, during training, the 
causal mask makes it possible to compute losses for every sub-sequence simultaneously,
\begin{align}
  \text{<sos>},\boldsymbol{p} & \rightarrow \ell_0  \rightarrow loss(\ell_0, \text{<sep>}),\\
  \text{<sos>},\boldsymbol{p}, \text{<sep>} & \rightarrow \ell_1  \rightarrow loss(\ell_1, t_1),\\
  \text{<sos>}, \boldsymbol{p}, \text{<sep>}, t_1  & \rightarrow \ell_2 \rightarrow loss(\ell_2, t_2), \\
        : & : \\
  \text{<sos>}, \boldsymbol{p}, \text{<sep>}, t_1,\cdots, t_{k}  & \rightarrow \ell_{k+1}  \rightarrow loss(\ell_{k+1}, \text{<eos>}) .
\end{align}
Notice that the losses are computed on the tokens to be predicted as well as the separator token. 
    
In evaluation mode, the model is used *autoregressively*: the prompt sequence $\text{<sos>}, \boldsymbol{p}, \text{<sep>}$ is entered into the model, which predicts the next token, $y_1$. That token is appended to the current input sequence to form the next input sequence $\text{<sos>}, \boldsymbol{p}, \text{<sep>}, y_1$ and the procedure repeats until either the token $\text{<eos>}$ is predicted or the maximum allowed output sequence length is reached, whichever comes first.

In [16]:
importlib.reload(tnm)

# Note: The decoration @torch.jit.unused will cause the PyTorch scripting tool
#       to ignore the decorated function.

class TinyLM(mlp.Model):
    
    def __init__(self, 
                 vocab_size : int,
                 max_seq_len : int,
                 emb_dim : int,
                 layers : int,
                 heads : int,
                 ff_dim : int,
                 dropout : float,
                 pad : int,
                 sos : int,
                 eos : int,
                 sep : int,
                 prompt_mask : bool=False,
                 device : torch.device=torch.device(
                     'cuda' if torch.cuda.is_available() else 'cpu')):
        '''
    Arguments:
        
         vocab_size : int            Vocabulary size
         max_seq_len : int           Maximum sequence length
         emb_dim : int               Embedding dimension
         layers : int                Number of decoding layers
         heads : int                 Number of attention heads
         ff_dim : int                Feed-forward network dimension
         dropout : float             Dropout probability
         
         pad : int                   Pad code
         sos : int                   Start-of-sequence code
         eos : int                   End-fo-sequence code
         sep : int                   Separator code
         
         prompt_mask : bool          If true allow all prompt tokens 
                                     to attend to each other [False]
         device:  torch.device       Computational device
        '''
        
        super().__init__()

        self.pad : int = pad  # make sure jit sees th
        self.sos : int = sos
        self.eos : int = eos
        self.sep : int = sep

        self.max_len : int = max_seq_len
        self.prompt_mask : bool = prompt_mask
        self.device : torch.device = device
        
        self.decoder = Decoder(
            vocab_size, max_seq_len, emb_dim, layers, heads, ff_dim, dropout,
            device)

    def make_mask(self, sequence: torch.Tensor) -> torch.Tensor:
        # sequence: [batch_size, seq_len]
        _, seq_len = sequence.shape
    
        pad_mask = (sequence != self.pad).unsqueeze(1).unsqueeze(2)
        # pad_mask: [batch_size, 1, 1, seq_len]

        sub_mask = torch.tril(
            torch.ones((seq_len, seq_len), device=self.device, dtype=torch.bool))
        # sub_mask: [seq_len, seq_len]

        # logical AND of the two masks
        mask = pad_mask & sub_mask
        # mask: [batch_size, 1, seq_len, seq_len]

        # If requested, allow all prompt tokens to attend to
        # each other by placing True at every prompt token. 
        if self.prompt_mask:
            is_sep = sequence == self.sep # find separators
            
            prompt_mask = ~torch.logical_xor(
                is_sep.cumsum(dim=-1)>=1, is_sep).unsqueeze(1).unsqueeze(2)
            
            mask = mask | prompt_mask
            
        return mask

    @torch.jit.unused
    def make_loss_mask(self, sequence: torch.Tensor) -> torch.Tensor:
        """
        Mask with 1 at positions at which loss should be computed, 
        0 elsewhere. Losses are computed for the separator token and
        all positions thereafter, but excluding the <pad> tokens.
    
        Arguments:
            sequence: [batch_size, seq_len]   A tensor of integers
    
        Returns:
            mask:     [batch_size, seq_len]   A float tensor, values in {0.0, 1.0}
        """

        # Place True at every <sep> token in the batch.
        is_sep = sequence == self.sep
        
        # Place True at <sep> and target tokens in the batch.
        sep_and_target = is_sep.cumsum(dim=-1) >= 1

        # Place True at every <pad>
        not_pad = sequence != self.pad

        # AND the two masks and convert to floats
        return (sep_and_target & not_pad).float()

    def forward(self, sequence : torch.Tensor) -> torch.Tensor:
        # sequence: [batch_size, seq_len]
       
        mask  = self.make_mask(sequence)
        # mask: [batch_size, 1, seq_len, seq_len]
        
        logits = self.decoder(sequence, mask)
        # logits: [batch_size, seq_len, vocab_size]

        return logits

    @classmethod
    @torch.jit.unused
    def from_config(cls, config: mlp.Config) -> 'TinyLM':
        """Convenience constructor for the training notebook."""
        
        return cls(
            vocab_size = config('VOCAB_SIZE'),
            max_seq_len= config('MAX_SEQ_LEN'),
            emb_dim    = config('EMB_DIM'),
            layers     = config('LAYERS'),
            heads      = config('HEADS'), 
            ff_dim     = config('FF_DIM'), 
            dropout    = config('DROPOUT'),
            pad        = config('PAD'),
            sos        = config('SOS'),
            eos        = config('EOS'),
            sep        = config('SEP'),
            prompt_mask= config('PROMPT_MASK', False),
            device     = config('DEVICE', 
                                torch.device(
                                    'cuda' if torch.cuda.is_available() \
                                    else 'cpu')) 
        )

In [17]:
def print_mask_check(sequence, mask, vocab):
    """vocab: dict mapping token id -> string token
    By Claude
    """
    for b in range(sequence.size(0)):
        tokens = [vocab[i.item()] for i in sequence[b]]
        values = [str(int(m.item())) for m in mask[b]]
        print("  ".join(f"{t:>6}" for t in tokens))
        print("  ".join(f"{v:>6}" for v in values))
        print()

## Training the Tiny Language Model

Our model is miniscule compared with the transformer models used today. Indeed, the model is small enough to be trained on a single GPU in just over an hour!
<img src="./logits_grid.png" align="left" width="500px"/>
In Section *Transformer Model* we advertised a key feature of this model: it processes all tokens in a sequence, $\boldsymbol{X}$, in parallel and computes logits for the next token for every sub-sequence. Consider our example of a sequence of size $k = 5$ as depicted in the figure to the left. The model outputs a logit for every input token and every token in the vocabulary as illustrated by the 2D tensor on the left of the figure in which the sequence is arranged from top to bottom. The size of the vocabulary in the figure is $|\mathbb{V}|=7$. The logits associated with each token are used to predict the *next* token. Therefore, we need to ensure that the logits tensor on the left, which is the output of the model, is correctly aligned with the target tensor on the right.
Since there's nothing to predict before `<sos>`, we slice off that token from the target tensor. Likewise, there's nothing to predict after `<eos>`, so we slice off the `<eos>` token from the logits tensor.  Now that the logit and target tensors are correctly aligned in the sense that the prediction derived from a given series of logits can now be compared with the target token at the same position,  we can compute the loss for each token, which is associated with a given sub-sequence.
<br clear="left"/>

### Loss function
A transformer is a multi-class classifier: the logits, when converted to probabilities, are used to decide which token comes next. Like most multi-class classifiers, a transformer is trained using the **cross entropy loss**. Given the data $(\ell_i, t_i)$, where $\ell_i$ is a vector of logits and $t_i$ is the true token at the $i^\text{th}$ position, the loss is given by
\begin{align}
    loss(\ell_{i, k}, t_i) & = - \log p_k, \quad\text{ with } k = t_i, \\
        p_k & = \texttt{softmax}_k(\ell_i) \equiv \frac{\exp(\ell_{i, k})}{\sum_{j} \exp(\ell_{i,j})},
\end{align}
and $\ell_{i, k}$ denotes the $k^\text{th}$ component of the logit vector $\ell_i$. The **empirical risk**, that is, average loss, is computed over a batch of sequences for all tokens *after* the `<sep>` token in each sequence, but excluding the `<pad>` tokens.

In [18]:
def train(objective, optimizer, scheduler, monitor,
          train_loader, train_small_loader, val_loader):

    for sequence in train_loader:
        
        objective.train()
        
        R = objective(sequence)
        
        optimizer.zero_grad()     # zero gradients
        
        R.backward()              # compute gradients

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)

        optimizer.step()          # make a single step in average loss

        # check whether to update learning rate
        scheduler.step()

        if monitor.step():

            # set mode to evaluation so that training-specific
            # operations such as dropout, etc., are disabled.
            objective.eval()

            seq = next(iter(train_small_loader))
            t_loss = objective(seq).item()
 
            seq = next(iter(val_loader))
            v_loss = objective(seq).item()

            # return current learning rate
            lr = scheduler.lr()

            # update loss file
            monitor(t_loss, v_loss, lr)

In [19]:
importlib.reload(tnm)

config('DEVICE', DEVICE)
model = TinyLM.from_config(config).to(DEVICE)
print(model)
print(f'The model has {mlp.number_of_parameters(model)} trainable parameters')
            
optimizer = torch.optim.Adam(model.parameters(), lr=config('base_lr'))

averageloss = nn.CrossEntropyLoss(reduction='none')
# --------------------------------------------------------
# Make a specialized objective from mlp.Objective
# --------------------------------------------------------
class TLMObjective(mlp.Objective):
    
    def __init__(self, model, avgloss):

        super().__init__(model, avgloss)
        
    def forward(self, sequence):
        # sequence[batch_size, seq_len]

        # Run the model to compute, in parallel, a vector 
        # of logits for every token in the sequence. The
        # logits for a given token are used to predict
        # the next token.
        logits = self.model(sequence)
        # logits: [batch_size, seq_len, vocab_size]
        batch_size, seq_len, vocab_size = logits.shape
        
        # Slice off the <eos> token from the logits tensor because
        # there's nothing to predict after the end-of-sequence.
        logits_  = logits[:, :-1, :]
        # logits_: [batch_size, seq_len-1, vocab_size]

        # Slice off the <sos> token from the sequence because
        # there are no logits for the start-of-sequence!
        targets  = sequence[:, 1:]
        # targets: [batch_size, seq_len-1]

        # Compute a loss for every token, but do not reduce!
        # Yes, there is wasted computation, but this makes
        # the code cleaner.
        loss_per_token = self.avgloss(
                logits_.reshape(-1, vocab_size), # Flatten
                targets.reshape(-1),             # Flatten
        ).reshape(batch_size, -1)  # Back to shape [batch_size, seq_len-1]

        # The logits and targets are now aligned. Create a mask 
        # to exclude the whole of the delimited prompt and all 
        # remaining pads after the separator token.
        loss_mask = self.model.make_loss_mask(sequence[:, :-1])

        # Use loss_mask to zero out all losses except the relevant ones. 
        # Then average to get the empirical risk, R
        R = (loss_per_token * loss_mask).sum() / loss_mask.sum()

        return R

objective = TLMObjective(model, averageloss)

# Instantiate learning rate step scheduler
scheduler = mlp.LRStepScheduler(
    optimizer,  
    n_steps=config('n_steps'), 
    n_iters_per_step=config('n_iters_per_step'), 
    base_lr=config('base_lr'), 
    gamma=config('gamma')
)

# Instantiate object that saves average losses to
# a csv file for realtime monitoring, as well as
# the model with the lowest average loss
monitor = mon.Monitor(
    config('n_iterations'),
    config('file/losses'),
    monitorstep=config('monitor_step'),
    newlossfile=True,
    frac=config('frac'),
    model=model,
    paramsfile=config('file/params'), 
    use_tensorboard=False
)

TinyLM(
  (decoder): Decoder(
    (tok_embedding): Embedding(40, 72)
    (pos_embedding): Embedding(128, 72)
    (layers): ModuleList(
      (0-2): 3 x DecoderLayer(
        (self_attention): MultiHeadAttention(
          (A): Linear(in_features=72, out_features=72, bias=False)
          (B): Linear(in_features=72, out_features=72, bias=False)
          (C): Linear(in_features=72, out_features=8, bias=False)
          (V): Linear(in_features=72, out_features=72, bias=True)
          (O): Linear(in_features=72, out_features=72, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (self_attention_norm): LayerNorm((72,), eps=1e-05, elementwise_affine=True)
        (feedforward): Feedforward(
          (linear_1): Linear(in_features=72, out_features=128, bias=True)
          (linear_2): Linear(in_features=128, out_features=72, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (silu): SiLU()
        )
        (feedforward_norm): LayerNorm((72

Note: 134,852 parameters with biases in Keys

In [20]:
print(f'\n\tComputational device: {str(DEVICE):s}\n')

TRAIN = False

if TRAIN:

    monitor.start()

    train(
        objective, optimizer, scheduler, monitor,
        train_loader, train_loader_val, val_loader
    )

    monitor.end()


	Computational device: cpu



## Using the Model

The test data are already tokenized, coded, and bracketed. Given a trained model and a prompt, generate output autoregressively.

In [21]:
def generate(model, 
             prompt: torch.Tensor, 
             eos: int, 
             pad: int,
             max_len: int, 
             device: torch.device) -> torch.Tensor:
    '''
    Given a model and a prompt, generate output. 
    '''

    if prompt.dim() != 2:
        prompt = prompt.view(1, -1)
    # prompt: [1, prompt_len]
    
    sequence = prompt.to(device)
    _, prompt_len = sequence.shape
    
    output = []

    for _ in range(max_len - prompt_len):

        # Execute model. The model computes logits for every
        # sub-sequence of the current sequence. But we really
        # need only the logits for the last sub-sequence, i.e.,
        # for the current sequence.
        logits    = model(sequence)
        # logits: [1, seq_len, vocab_size]
        
        # Get logits of last sub-sequence, i.e., the current sequence.
        last      = logits[0, -1, :]
        # last: [seq_len, vocab_size]

        # Convert logits to probabilities.
        probs     = torch.softmax(last, dim=-1)
        # probs: [seq_len, vocab_size]
        
        # Choose the most probable token.
        _, code_t = torch.topk(probs, k=1)
        code      = int(code_t[0].item())

        if code == pad: continue
        if code == eos: break

        # Add next token to current input sequence and iterate
        next_token = torch.full((1, 1), code, dtype=torch.long, device=device)
        sequence   = torch.cat([sequence, next_token], dim=-1)
        output.append(code)

    return torch.tensor(output, dtype=torch.long)

In [22]:
importlib.reload(mlp)
# ----------------------------------------------------------------
# Example of a model with ~99% accuracy
config_filename = 'runs/TLM/TLM_config.yaml'
#config_filename = config('file/config')

config = mlp.Config(config_filename)
print('\tCONFIGURATION\n')
print(config)

# Initialize model to the best-fit parameters
model = TinyLM.from_config(config).to(DEVICE)
model.load(config('file/params'))
print('\tMODEL\n')
print(model)
print(f'Number of parameters: {mlp.number_of_parameters(model)}')

# Save best model
print(f'\nSave fully-loaded model to {config("file/script")}')
torch.jit.script(model).save(config('file/script'))

	CONFIGURATION

name: TLM
file:
  config: runs/TLM/TLM_config.yaml
  losses: runs/TLM/TLM_losses.csv
  params: runs/TLM/TLM_params.pth
  script: runs/TLM/TLM_script.pth
  init_params: runs/TLM/TLM_init_params.pth
  plots: runs/TLM/TLM_plots.png
train_size: 12000
val_size: 156
test_size: 800
batch_size: 32
monitor_step: 100
frac: 0.01
n_steps: 16
n_iterations: 800000
n_iters_per_step: 50000
base_lr: 0.0016
gamma: 0.8312378961427878
DATAFILE: ../data/seq2seq_series_2terms.txt
MAX_SEQ_LEN: 128
EMB_DIM: 72
LAYERS: 3
HEADS: 8
FF_DIM: 128
DROPOUT: 0.1
VOCAB_SIZE: 40
PAD: 0
SOS: 1
EOS: 2
SEP: 3
PROMPT_MASK: false



RuntimeError: Error(s) in loading state_dict for TinyLM:
	Missing key(s) in state_dict: "decoder.layers.0.self_attention.A.weight", "decoder.layers.0.self_attention.B.weight", "decoder.layers.0.self_attention.C.weight", "decoder.layers.1.self_attention.A.weight", "decoder.layers.1.self_attention.B.weight", "decoder.layers.1.self_attention.C.weight", "decoder.layers.2.self_attention.A.weight", "decoder.layers.2.self_attention.B.weight", "decoder.layers.2.self_attention.C.weight". 
	Unexpected key(s) in state_dict: "decoder.layers.0.self_attention.Q.weight", "decoder.layers.0.self_attention.Q.bias", "decoder.layers.0.self_attention.K.weight", "decoder.layers.0.self_attention.K.bias", "decoder.layers.1.self_attention.Q.weight", "decoder.layers.1.self_attention.Q.bias", "decoder.layers.1.self_attention.K.weight", "decoder.layers.1.self_attention.K.bias", "decoder.layers.2.self_attention.Q.weight", "decoder.layers.2.self_attention.Q.bias", "decoder.layers.2.self_attention.K.weight", "decode

In [23]:
# Load saved model
model = torch.jit.load(config('file/script'))
model.eval()

PRINT_MISTAKES = False
M = 0
F = 0.0
T = 10
n_mistakes = 100
k_mistakes = 0

EOS = config('EOS')
PAD = config('PAD')
MAX_LEN = config('MAX_SEQ_LEN')

for i, sequence in enumerate(test_loader):
    # sequence: [1, seq_len]

    # Extract prompt and target
    prompt, target = seqdata.split(sequence)
    target_= seqdata.str(target)

    # Generate output sequence
    output = generate(
        model, prompt, EOS, PAD, MAX_LEN, DEVICE)

    output_= seqdata.str(output)

    # count how often we're right
    if output_ == target_:
        M += 1
        F = M / (i+1)
    else:
        if PRINT_MISTAKES:
            print()
            print(f'{i:3d} true: {target_}')
            print(f'    pred: {output_}')
            print('-'*80)
            k_mistakes += 1
            PRINT_MISTAKES = k_mistakes < n_mistakes
            
    if i % T == 0:
        print(f'\r{i:8d}\taccuracy: {F:8.3f}', end='')
        
N  = len(test_data)
dF = np.sqrt(F*(1-F)/N)
print()
print(f'Accuracy: {F:8.3f} +/- {dF:.3f}')

     790	accuracy:    0.994
Accuracy:    0.993 +/- 0.003
